In [1]:
import numpy as np
import jax.numpy as jnp
import optax
from flax import nnx
import galax.potential as gp
from galactoPINNs.data import generate_static_data, scale_data
from galactoPINNs.models.static_model import StaticModel
from galactoPINNs.train import train_model_static

true_potential = gp.MilkyWayPotential()      # our stand-in for "reality"


In [2]:
raw = generate_static_data(
    galax_potential=true_potential,
    n_samples_train=1000, n_samples_test=4096,
    r_max_train=25, r_max_test=25,
)


In [3]:
halo_rs = 15.62
ab_pot = gp.NFWPotential(m=5.4e11, r_s=halo_rs, units="galactic")
scale_config = {"r_s": halo_rs, "include_analytic": True, "ab_potential": ab_pot}
scaled, transformers = scale_data(raw, scale_config)

x_train, a_train = scaled["x_train"], scaled["a_train"]
x_val,   a_val   = scaled["x_val"],   scaled["a_val"]


In [4]:
# Sightlines run from the Sun (observer) to each point. n_vecs is a UNIT vector
# (dimensionless), so it's identical in physical or scaled space — compute it from
# the physical positions in `raw`. These are the SAME 1000 points Model A trains on,
# so the only difference between the two models is full-vector vs projection.
observer = np.array([-8.1, 0.0, 0.0])                       # Sun, galactocentric kpc
dx = np.asarray(raw["x_train"]) - observer                  # observer -> point
n_vecs = jnp.asarray(dx / np.linalg.norm(dx, axis=1, keepdims=True))   # (N, 3)
print("n_vecs", n_vecs.shape, "mean|n̂| =", float(jnp.linalg.norm(n_vecs, axis=1).mean()))  # ~1.0


n_vecs (1000, 3) mean|n̂| = 1.0


In [5]:
train_config = {
    "x_transformer": transformers["x"], "a_transformer": transformers["a"],
    "u_transformer": transformers["u"], "r_s": halo_rs,
    "include_analytic": True, "scale": "nfw", "depth": 6,
}
# Same seed => identical initial weights => the ONLY difference is the training signal.
netA = StaticModel(train_config, rngs=nnx.Rngs(0))     # will see full 3-vectors
netB = StaticModel(train_config, rngs=nnx.Rngs(0))     # will see LOS scalars only


In [6]:
tx = optax.adam(1e-3)
outA = train_model_static(netA, tx, x_train, a_train, 2000, log_every=500)


In [7]:
outB = train_model_static(
    netB, optax.adam(1e-3), x_train, a_train, 2000,
    line_of_sight=True, n_vecs=n_vecs, lambda_rel=0.0,   # pure absolute LOS error
    log_every=500,
)
